# DuckPD Time-Series Embeddings: End-to-End Walkthrough

This notebook builds deterministic, model-free time-series representations entirely in DuckDB. It covers ordered grouped windows, representation contracts, lazy `embed_series()` execution, raw-window `search_series()`, reusable typed queries, and metadata-preserving Parquet persistence.

### What you will learn
- Why time-series representations require explicit row order and fixed-size windows.
- How to combine multiple semantic channels into one stable vector space.
- How centering, channel ordering, flattening, and unit normalization are declared.
- How warm-up rows, nulls, and zero-scale windows are handled.
- How raw channel windows become exact, representation-aware search queries.
- How representation fingerprints reject equal-width incompatible spaces before execution.
- How to persist and restore representation metadata without a model runtime.

## 1. Imports and session

The example is self-contained and offline. NumPy and pandas only create a small deterministic input fixture; all windowing, representation compilation, normalization, and retrieval run through DuckPD and DuckDB.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pandas

import duckpd as pd

session = pd.connect(memory_limit="512MB", threads=4)
print(f"DuckPD version: {pd.__version__}")
print(f"Executions: {session.execution_count}")

DuckPD version: 0.1.4
Executions: 0


## 2. Create deterministic multi-asset market data

Each ticker receives its own chronological price series. The synthetic fixture deliberately contains different trends and oscillations so nearest-window retrieval has meaningful variation while remaining reproducible.

In [2]:
rng = np.random.default_rng(20260910)
tickers = ("NVDA", "AMD", "INTC")
bars_per_ticker = 72
records = []

for ticker_number, ticker in enumerate(tickers):
    timestamps = pandas.date_range("2026-01-05 09:30", periods=bars_per_ticker, freq="min")
    phase = np.linspace(0.0, 5.0 * np.pi, bars_per_ticker) + ticker_number * 0.7
    innovations = rng.normal(0.0, 0.0015, bars_per_ticker)
    returns = 0.0012 * np.sin(phase) + innovations
    opens = (100.0 + ticker_number * 35.0) * np.exp(np.cumsum(returns))
    closes = opens * (1.0 + returns)
    spreads = opens * (0.0015 + 0.001 * np.abs(np.cos(phase)))
    highs = np.maximum(opens, closes) + spreads
    lows = np.minimum(opens, closes) - spreads
    records.extend(
        zip(timestamps, [ticker] * bars_per_ticker, opens, highs, lows, closes, strict=True)
    )

market_data = pandas.DataFrame(
    records, columns=["timestamp", "ticker", "open", "high", "low", "close"]
)
market_data.head()

,timestamp,ticker,open,high,low,close
0,2026-01-05 09:30:00,NVDA,100.005596,100.261206,99.755582,100.011192
1,2026-01-05 09:31:00,NVDA,99.965462,100.212940,99.677860,99.925337
2,2026-01-05 09:32:00,NVDA,100.283181,100.842454,100.042131,100.601404
3,2026-01-05 09:33:00,NVDA,100.303752,100.553791,100.074286,100.324325
4,2026-01-05 09:34:00,NVDA,100.442534,100.795689,100.228258,100.581413


## 3. Create a lazy frame with an explicit order contract

A rolling time-series operation is undefined without order. `order_by=["ticker", "timestamp"]` declares the deterministic sequence used within each ticker. Constructing the frame copies the small fixture into the session but does not execute the analytical plan.

In [3]:
prices = session.from_pandas(
    market_data,
    order_by=["ticker", "timestamp"],
)

print(prices)
print(f"Executions after frame construction: {session.execution_count}")

DuckPD DataFrame
Columns: ['timestamp', 'ticker', 'open', 'high', 'low', 'close']
Plan: SortPlan
Executions after frame construction: 0


## 4. Engineer semantic channels lazily

A representation contract names channels by meaning, not by storage-column label. Here `bar_return` captures direction and `intrabar_range` captures relative volatility. Both expressions remain inside the lazy plan.

In [4]:
features = prices.assign(
    bar_return=lambda frame: (frame["close"] - frame["open"]) / frame["open"],
    intrabar_range=lambda frame: (frame["high"] - frame["low"]) / frame["open"],
)

features[["open", "close", "bar_return", "intrabar_range"]].head(6)

,open,close,bar_return,intrabar_range
0,135.027766,135.055535,0.000206,0.004735
1,135.123802,135.219873,0.000711,0.004921
2,135.684262,136.245882,0.004139,0.007970
3,136.037613,136.391424,0.002601,0.006012
4,136.025555,136.013497,-0.000089,0.003117
5,135.927407,135.829295,-0.000722,0.004188


## 5. Build complete grouped rolling windows

`rolling(WINDOW).to_array()` is DuckPD's fixed-window extension. It emits nullable `FLOAT[WINDOW]` arrays in oldest-first order. Grouping isolates ticker histories. The first `WINDOW - 1` rows in every ticker are null warm-up rows; DuckPD never pads an incomplete window.

In [5]:
WINDOW = 8

windows = features.assign(
    return_window=lambda frame: frame.groupby("ticker")["bar_return"].rolling(WINDOW).to_array(),
    range_window=lambda frame: frame.groupby("ticker")["intrabar_range"].rolling(WINDOW).to_array(),
)

print(f"Window columns: return_window and range_window are FLOAT[{WINDOW}]")
print(f"Executions after window planning: {session.execution_count}")

Window columns: return_window and range_window are FLOAT[8]
Executions after window planning: 1


## 6. Declare the representation space

The immutable specification is the identity of the vector space:

1. Each channel is centered independently within its window.
2. Channels are flattened in declared order, each oldest-first.
3. The concatenated vector is normalized to unit length.
4. A zero-length normalized vector becomes null instead of silently producing invalid numbers.

The output dimension is `window × channel_count`. Changing any semantic field changes the SHA-256 fingerprint.

In [6]:
representation = pd.series_representation(
    window=WINDOW,
    channels=("bar_return", "intrabar_range"),
    sampling="observations",
    data_contract="tutorial/ohlc-shape/v1",
    normalization="center",
    unit_norm=True,
    zero_scale="null",
)

print(f"Dimension: {representation.dimension}")
print(f"Fingerprint: {representation.fingerprint}")
representation.to_dict()

Dimension: 16
Fingerprint: 5f088c8a6e7686ca311c2232eeeb801e5f51dedcb625fbe043b2095d7d21b70b


{'schema_version': 1,
 'window': 8,
 'channels': ['bar_return', 'intrabar_range'],
 'sampling': 'observations',
 'data_contract': 'tutorial/ohlc-shape/v1',
 'step': None,
 'normalization': 'center',
 'unit_norm': True,
 'zero_scale': 'null',
 'encoder': None,
 'output_type': 'FLOAT',
 'dimension': 16,
 'layout': 'channel-major-oldest-first-v1'}

## 7. Compile native embeddings lazily

`columns` maps semantic channel names to physical window columns. Mapping insertion order does not control layout; `representation.channels` does. Native representations run as DuckDB expressions—no Python row loop, model download, or inference runtime.

In [7]:
embedded = windows.embed_series(
    columns={
        "intrabar_range": "range_window",
        "bar_return": "return_window",
    },
    into="market_shape",
    representation=representation,
)

print(f"Output dtype: FLOAT[{representation.dimension}]")
print(f"Executions after embed_series planning: {session.execution_count}")

explanation = json.loads(embedded.explain(mode="json"))
explanation["execution_boundaries"]["embedding_operations"]

Output dtype: FLOAT[16]
Executions after embed_series planning: 1


[{'operation': 'embed_series',
  'backend': 'native',
  'representation_fingerprint': '5f088c8a6e7686ca311c2232eeeb801e5f51dedcb625fbe043b2095d7d21b70b',
  'dimension': 16,
  'normalization': 'center',
  'unit_norm': True,
  'batch_size': 256,
  'null_policy': 'propagate',
  'channels': ['bar_return', 'intrabar_range'],
  'boundary': 'duckdb_native_expression',
  'persistence': 'lazy'}]

## 8. Inspect warm-up and embedded rows

The first seven rows per ticker remain null because they do not contain eight observations. A non-null result is always one complete `FLOAT[16]` vector. `null_policy="propagate"` is the default: any null input window makes the whole representation null. Use `null_policy="error"` when null windows should abort execution instead.

In [8]:
nvda_preview = embedded[embedded["ticker"] == "NVDA"][
    ["bar_return", "intrabar_range", "market_shape"]
].head(WINDOW + 2)
nvda_preview

,bar_return,intrabar_range,market_shape
0,0.000056,0.005056,<NA>
1,-0.000401,0.005353,<NA>
2,0.003173,0.007981,<NA>
3,0.000205,0.004781,<NA>
4,0.001383,0.005649,<NA>
5,0.000157,0.004053,<NA>
6,-0.001046,0.004528,<NA>
7,0.001752,0.004797,"[-0.12610143, -0.22160421, 0.5248464, -0.09495..."
8,0.001311,0.004707,"[-0.25369218, 0.49078062, -0.12737942, 0.11786..."
9,0.001771,0.005587,"[0.44453558, -0.18830827, 0.06276176, -0.19856..."


## 9. Create a raw query from one observed window

For this tutorial, the channel windows behind the first complete NVDA representation are the query-by-example. `search_series()` freezes these raw observations during planning and applies the same representation recipe only when the search executes.

Selecting the query row is intentionally eager; query representation and corpus search remain lazy until `collect()`.

In [9]:
nvda_candidates = embedded[(embedded["ticker"] == "NVDA") & embedded["market_shape"].notna()]
query_row = nvda_candidates[["timestamp", "return_window", "range_window"]].head(1)
query = {
    "bar_return": tuple(float(value) for value in query_row.iloc[0]["return_window"]),
    "intrabar_range": tuple(float(value) for value in query_row.iloc[0]["range_window"]),
}

print(f"Query endpoint: {query_row.iloc[0]['timestamp']}")
print(f"Query observations per channel: {WINDOW}")
print(f"Executions after extracting the query: {session.execution_count}")

Query endpoint: 2026-01-05 09:37:00
Query observations per channel: 8
Executions after extracting the query: 3


## 10. Run exact representation-aware retrieval

`search_series()` resolves verified metadata from `market_shape`, applies the declared native representation to the raw query, and ranks complete NVDA windows by L2 distance. Because every vector has unit norm, L2 and cosine produce equivalent ordering here. `tie_breaker="timestamp"` makes equal-distance results deterministic. The query window itself appears first at distance zero.

In [10]:
matches = nvda_candidates.vector.search_series(
    query,
    column="market_shape",
    representation=representation,
    metric="l2",
    k=6,
    tie_breaker="timestamp",
)[["timestamp", "close", "bar_return", "intrabar_range", "_distance"]]

print(f"Executions before collecting search: {session.execution_count}")
match_result = matches.collect()
assert match_result.iloc[0]["_distance"] == 0.0
match_result

Executions before collecting search: 3


,timestamp,close,bar_return,intrabar_range,_distance
0,2026-01-05 09:37:00,100.705423,0.001752,0.004797,0.000000
1,2026-01-05 10:09:00,101.192239,0.002680,0.007079,0.714703
2,2026-01-05 10:16:00,101.456303,-0.000792,0.005253,1.014481
3,2026-01-05 09:52:00,100.573812,0.000010,0.003319,1.021515
4,2026-01-05 10:31:00,100.957555,0.001737,0.005934,1.022098
5,2026-01-05 10:18:00,101.519690,0.000219,0.003954,1.051438


## 11. Prove incompatible spaces fail before execution

Dimensions alone are insufficient. The following specification has the same shape but a different data contract. Passing it to `search_series()` is a compatibility assertion, not an override, so DuckPD rejects it during planning without corpus execution.

In [11]:
incompatible_representation = pd.series_representation(
    window=WINDOW,
    channels=("bar_return", "intrabar_range"),
    sampling="observations",
    data_contract="tutorial/different-contract/v1",
    normalization="center",
    unit_norm=True,
    zero_scale="null",
)
executions_before_rejection = session.execution_count

try:
    nvda_candidates.vector.search_series(
        query,
        column="market_shape",
        representation=incompatible_representation,
    )
except pd.errors.UnsupportedOperationError as error:
    print(f"Rejected as expected: {error}")

assert session.execution_count == executions_before_rejection

Rejected as expected: search_series requires series metadata matching the requested representation


## 12. Persist vectors and reuse an eager typed query

`write_parquet()` streams the lazy result directly from DuckDB. DuckPD writes a managed sidecar containing the representation contract. `Session.embed_series_query()` eagerly applies the same native recipe once and returns an immutable fingerprinted query. Reloading the Parquet file restores the column identity, so that typed query remains valid.

In [12]:
DEMO_DIR = Path("demo") if Path("demo").is_dir() else Path(".")
OUTPUT_DIR = DEMO_DIR / ".tmp"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT = OUTPUT_DIR / "time-series-embeddings.parquet"

embedded.write_parquet(OUTPUT, overwrite=True)
restored = session.read_parquet(OUTPUT)
typed_query = session.embed_series_query(query, representation=representation)
restored_matches = (
    restored[(restored["ticker"] == "NVDA") & restored["market_shape"].notna()]
    .vector.search(
        typed_query,
        column="market_shape",
        metric="l2",
        k=3,
        tie_breaker="timestamp",
    )[["_distance"]]
    .collect()
)

assert restored_matches.iloc[0]["_distance"] == 0.0
print(f"Persisted data: {OUTPUT}")
print(f"Persisted metadata: {OUTPUT}.duckpd-embeddings.json")
print("Reusable typed search succeeded after reload.")

Persisted data: demo/.tmp/time-series-embeddings.parquet
Persisted metadata: demo/.tmp/time-series-embeddings.parquet.duckpd-embeddings.json
Reusable typed search succeeded after reload.


## 13. Cleanup and production checklist

For production workloads:

- Declare the true chronological order before rolling.
- Partition windows by every entity boundary that must not leak history.
- Treat `data_contract` as a versioned semantic schema.
- Persist reusable representations instead of recomputing them for repeated searches.
- Use raw channel mappings with `search_series()` or reusable fingerprinted queries from `Session.embed_series_query()`.
- Expect warm-up rows to be null; do not pad incomplete history.
- Native recipes support deterministic normalization and need no model runtime. Learned series encoders are intentionally separate and are not used here.

In [13]:
session.close()
OUTPUT.unlink(missing_ok=True)
Path(f"{OUTPUT}.duckpd-embeddings.json").unlink(missing_ok=True)
print("Session closed and tutorial artifacts removed.")

Session closed and tutorial artifacts removed.
